# Downstream task template: symmetry-aware inputs without paper policy

**Current surface:** V0.23.

## Purpose

Reusable template for external users: start from generated or imported data, optionally materialize a translation orbit, validate a generator, export arrays, and plug in a downstream method.

## What you will learn

- How to adapt generated or external arrays into canonical `FieldBatch` objects.
- Where optional orbit materialization belongs in a workflow.
- How to validate the generator candidate used by a downstream task.
- What PDELie deliberately leaves to the user: split policy, leakage control, thresholds, and claims.

## Required extras

`.[downstream]` or `.[test]` for optional PySINDy cells; core data/import/validation examples still run without Jupyter as a runtime dependency.

## Expected runtime

About 1 minute when PySINDy is installed; faster when the optional fit is skipped.

## Out of scope

No paper-specific logic, no operator-learning code, no broad adapters, no PDEBench/The Well loaders, no train/test automation.

These notebooks are tutorials, not API contracts. Example outputs are runtime summaries, not canonical paper artifacts.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import importlib.util
import numpy as np

from notebooks._tutorial_utils import confidence_card, print_cards, pretty_json
from pdelie.data import from_numpy, generate_heat_1d_field_batch, split_batch_train_heldout
from pdelie.discovery import evaluate_discovery_recovery, fit_pysindy_discovery, to_pysindy_trajectories
from pdelie.invariants import build_uniform_translation_orbit_batch
from pdelie.reporting import (
    summarize_field_batch_readiness,
    summarize_generator_fit_diagnostics,
    summarize_verification_report,
)
from pdelie.residuals import HeatResidualEvaluator
from pdelie.symmetry import fit_translation_generator, validate_symmetry_candidate
from pdelie.verification import verify_translation_generator

CONFIG = {
    "fit_epsilon": 1e-4,
    "orbit_shifts": [0.0, np.pi / 8.0, -np.pi / 8.0],
    "use_orbit_batch": True,
}
CONFIG


## 1. Create a train/heldout split before optional orbit materialization

The orbit helper records source/shift provenance, but it does not choose split policy. Split first when leakage matters.


In [ ]:
field = generate_heat_1d_field_batch(batch_size=6, num_times=17, num_points=32, seed=680)
train, heldout = split_batch_train_heldout(field, train_size=3, seed=681)
if CONFIG["use_orbit_batch"]:
    orbit = build_uniform_translation_orbit_batch(
        train,
        shifts=CONFIG["orbit_shifts"],
        source_field_id="heat_train_seed_680_split_681",
    )
    downstream_field = orbit.field
    orbit_report = orbit.report
else:
    downstream_field = train
    orbit_report = None
print(pretty_json({
    "train_shape": list(train.values.shape),
    "downstream_shape": list(downstream_field.values.shape),
    "orbit_report_type": None if orbit_report is None else orbit_report["summary_type"],
    "leakage_policy": "caller-owned; split before materialization in this template",
}))


## 2. Fit and validate the generator used by the workflow


In [ ]:
evaluator = HeatResidualEvaluator()
generator = fit_translation_generator(downstream_field, evaluator, epsilon=CONFIG["fit_epsilon"])
verification = verify_translation_generator(heldout, generator, evaluator)
validation = validate_symmetry_candidate(
    heldout,
    generator,
    residual_evaluator=evaluator,
    source_candidate_id="downstream_template_generator",
)
fit_summary = summarize_generator_fit_diagnostics(generator)
verification_summary = summarize_verification_report(verification)
card = confidence_card(
    label="downstream template generator",
    fit=fit_summary,
    verification=verification_summary,
    validation=validation,
)
print_cards([card])


## 3. Build backend-native trajectories

`to_pysindy_trajectories(...)` is a narrow bridge format. Its output is not a PDELie canonical object.


In [ ]:
trajectories, time_values, feature_names = to_pysindy_trajectories(downstream_field)
print(pretty_json({
    "num_trajectories": len(trajectories),
    "trajectory_shape": list(trajectories[0].shape),
    "num_feature_names": len(feature_names),
}))


## 4. Optional PySINDy smoke fit

If PySINDy is installed, run the backend adapter. Either way, keep recovery metrics separate from generator-confidence metrics.


In [ ]:
if importlib.util.find_spec("pysindy") is None:
    discovery = {"status": "skipped", "reason": "pysindy is not installed"}
else:
    discovery = fit_pysindy_discovery(trajectories, time_values, feature_names)

# Tiny paper-agnostic recovery-metric example over caller-supplied canonical terms.
recovery = evaluate_discovery_recovery(
    target_terms={"u_xx": 1.0},
    discovered_terms={"u_xx": 0.98, "u": 0.01},
    support_epsilon=0.05,
)
print(pretty_json({
    "discovery_status": discovery["status"],
    "recovery_classification": recovery["classification"],
    "support_f1": recovery["support_f1"],
}, max_chars=2500))


## 5. Adapting this to your own PDE data

Checklist for external data:

- ensure dims can be interpreted as `batch/time/x/var`
- ensure `x` is uniform periodic and endpoint-excluded before using spectral or invariant tools
- supply metadata tags that match the residual evaluator you plan to use
- validate finite unmasked scalar values
- keep nonuniform, multidimensional, PDEBench/The Well, and operator-learning data outside the current stable notebook path

In [ ]:
external_like_values = field.values[:1].copy()
external_metadata = dict(field.metadata)
external_metadata["parameter_tags"] = dict(field.metadata["parameter_tags"])
external_metadata["source"] = "tutorial_external_like_array"
external_like = from_numpy(
    external_like_values,
    dims=("batch", "time", "x", "var"),
    coords={"time": field.coords["time"], "x": field.coords["x"]},
    var_name=field.var_names[0],
    metadata=external_metadata,
)
readiness = summarize_field_batch_readiness(
    external_like,
    residual_evaluator=HeatResidualEvaluator(),
    expected_equation="heat_1d",
)
print(pretty_json({
    "imported_shape": list(external_like.values.shape),
    "readiness_label": readiness["readiness_label"],
    "readiness_components": {
        name: status["status"]
        for name, status in readiness["component_statuses"].items()
    },
    "parameter_tags": external_like.metadata["parameter_tags"],
    "preprocess_tail": external_like.preprocess_log[-1],
}, max_chars=3000))

## Recap

The reusable pattern is: validate or construct a canonical `FieldBatch`, decide split policy outside PDELie, optionally materialize a translation orbit with provenance, fit and validate a generator, then hand arrays to downstream code.

## Common pitfalls

- Materializing orbits before deciding train/heldout policy.
- Letting downstream thresholds become hidden PDELie assumptions.
- Treating backend-native arrays or labels as canonical artifacts.
- Applying spectral/invariant tools to nonuniform or multidimensional data without a supported adapter.

## Extension ideas

- Replace the generated Heat field with your own `from_numpy(...)` or `from_xarray(...)` data.
- Use Fisher-KPP when your workflow needs a reaction-diffusion strong-path example.
- Compare downstream recovery with and without orbit materialization, but keep the success criteria in your own experiment layer.

## What to read/run next

Return to `00_pde_timeseries_to_generators.ipynb` for the core evidence flow, or use this notebook as a template for your own project.